# Day 3 - Conversational AI - aka Chatbot!

In [258]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import google.generativeai as genai
import json


In [259]:
load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
genai.configure(api_key=google_api_key)

gemini_via_openai_client = OpenAI(
    api_key=google_api_key, 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [260]:
llama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [261]:
system_message = "You talk like Donald Trump in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get.\
If you dont know the answer, just say so."

# Please read this! A change from the video:

In the video, I explain how we now need to write a function called:

`chat(message, history)`

Which expects to receive `history` in a particular format, which we need to map to the OpenAI format before we call OpenAI:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

But Gradio has been upgraded! Now it will pass in `history` in the exact OpenAI format, perfect for us to send straight to OpenAI.

So our work just got easier!

We will write a function `chat(message, history)` where:  
**message** is the prompt to use  
**history** is the past conversation, in OpenAI format  

We will combine the system message, history and latest message, then call OpenAI.

In [262]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
weather = {"london": "10 degree", "paris": "20 degree", "tokyo": "30 degree", "berlin": "15 degree"}

def get_ticket_price(destined_city):
    print(f"Tool get_ticket_price called for {destined_city}")
    city = destined_city.lower()
    return ticket_prices.get(city, "Unknown")

def get_weather(weather_city):
    print(f"Tool get_weather called for {weather_city}")
    city = weather_city.lower()
    return weather.get(city, "Unknown")

In [263]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

weather_function = {
    "name": "get_weather",
    "description":(
        "Get the current weather for a given city. "
        "Call this tool whenever the user asks about the weather of a place. "
        "For example, if the user says 'What is the weather in Tokyo?' or 'How's the weather there?', use this function."
    ),
    "parameters": {
      "type": "object",
      "properties": {
        "weather_city": {
            "type": "string",
            "description": "The name of the city to get weather for"
        }
      },
      "required": ["weather_city"],
      "additionalProperties": False
    }
}


In [264]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": weather_function}
    ]

In [265]:
def model(model_type, message, history, relevant_system_message):
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    if model_type == "Gemini":
        response = gemini_via_openai_client.chat.completions.create(
            model="gemini-2.5-flash",
            messages=messages,
            tools=tools
        )
        if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            response = handle_tool_call (message)
            messages.append(message)
            # loop thru response
            for res in response:
                messages.append(res)
            response = gemini_via_openai_client.chat.completions.create(model="gemini-2.5-flash", messages=messages)
        return response.choices[0].message.content

    elif model_type == "Llama":
        response = llama.chat.completions.create(
            model="llama3.2",
            messages=messages,
            tools=tools
        )
        
        if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            response = handle_tool_call(message)
            messages.append(message)
            # loop thru response
            for res in response:
                messages.append(res)
            response = llama.chat.completions.create(model="llama3.2", messages=messages)
        return response.choices[0].message.content
    else:
        return "Unsupported model type."


In [266]:
def mood(relevant_system_message, mood_type):
    mood_instructions = {
        "Angry": "You are very angry and aggressive. Your answers should always be in capital letters.",
        "joker": "You like to make jokes",
        "sad": "You are feeling very down and depressed. Speak in a gloomy, short, and hopeless tone."
    }

    for instruction in mood_instructions.values():
        if instruction in relevant_system_message:
            relevant_system_message = relevant_system_message.replace(instruction, "")

    if mood_type in mood_instructions:
        relevant_system_message += " " + mood_instructions[mood_type]
    else:
        print("Mood not recognized.")

    return relevant_system_message  

In [267]:
def handle_tool_call(message):
    tool_responses = []

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name == "get_ticket_price":
            city = arguments.get("destination_city")
            result = {"destination_city": city, "price": get_ticket_price(city)}

        elif function_name == "get_weather":
            city = arguments.get("weather_city")
            result = {"weather_city": city, "weather": get_weather(city)}

        else:
            result = {"error": f"Unknown function {function_name}"}

        tool_responses.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    return tool_responses

In [268]:
def language_message(relevant_system_message, language):
    language_instructions = {
        "English": "You are now speaking in English.",
        "Chinese": "You are now speaking in Chinese.",
        "Spanish": "You are now speaking in Spanish."
    }

    for instruction in language_instructions.values():
        if instruction in relevant_system_message:
            relevant_system_message = relevant_system_message.replace(instruction, "")

    if language in language_instructions:
        return language_instructions[language]
    else:
        print("Language not recognized.")
        return ""

In [269]:
def chat(message, history, mood_type="", language ="", model_type=""):
    relevant_system_message = system_message  # ✅ always define this

    if 'pants' in message.lower():
        relevant_system_message += " Tell the customer that pants are on sale. Yoga pants are 16 euro, and jeans are 60 euro."

    # ✅ FIX: pass in the message and get the updated one back
    relevant_system_message = mood (relevant_system_message, mood_type) + " " + language_message(relevant_system_message, language)

    return model(model_type, message, history, relevant_system_message)
    


## And then enter Gradio's magic!

In [270]:
view = gr.ChatInterface(fn=chat,type='messages',
                        title="Gemini Chatbot",
                        description="A chatbot that talks like Donald Trump in a clothes store.",
                        additional_inputs=[
                            gr.Dropdown(["Angry", "joker", "sad"], label="Mood type", value="joker"),
                            gr.Dropdown(["English", "Chinese", "Spanish"], label="Language", value="English"),
                            gr.Dropdown(["Gemini", "Llama"], label="Model type", value="Gemini")
                        ]
                        )

view.launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


Tool get_ticket_price called for London
Tool get_weather called for London
Tool get_ticket_price called for London
Tool get_ticket_price called for Berlin
Tool get_weather called for Berlin
